# 第二次作业：CNN实现SVHN图像分类任务

本 Notebook 按作业 PDF 要求完成：

- 读取 `train_32x32.mat` 和 `test_32x32.mat`
- 构建并训练 CNN 完成 SVHN Format 2 的 0~9 分类
- 输出测试准确率（Test Accuracy）
- 绘制训练/测试准确率曲线
- 绘制训练/测试损失曲线
- 保存最佳模型


In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [ ]:
# =========================
# 0. 全局设置
# =========================
SEED = 42
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
NUM_EPOCHS = 15   # 可先改成 5 试跑，正式训练再改回 15 或更高
NUM_CLASSES = 10
MODEL_SAVE_PATH = "svhn_cnn_model.pth"

TRAIN_MAT_PATH = "train_32x32.mat"
TEST_MAT_PATH = "test_32x32.mat"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
# =========================
# 1. 读取数据
# =========================
if not os.path.exists(TRAIN_MAT_PATH):
    raise FileNotFoundError(f"未找到文件: {TRAIN_MAT_PATH}")

if not os.path.exists(TEST_MAT_PATH):
    raise FileNotFoundError(f"未找到文件: {TEST_MAT_PATH}")

train_mat = loadmat(TRAIN_MAT_PATH)
test_mat = loadmat(TEST_MAT_PATH)

print("train_mat keys:", train_mat.keys())
print("test_mat keys:", test_mat.keys())

X_train_raw = train_mat["X"]   # shape: (32, 32, 3, N)
y_train_raw = train_mat["y"]   # shape: (N, 1)

X_test_raw = test_mat["X"]
y_test_raw = test_mat["y"]

print("X_train_raw shape:", X_train_raw.shape)
print("y_train_raw shape:", y_train_raw.shape)
print("X_test_raw shape:", X_test_raw.shape)
print("y_test_raw shape:", y_test_raw.shape)

In [ ]:
# =========================
# 2. 数据预处理
# =========================
y_train = y_train_raw.squeeze()
y_test = y_test_raw.squeeze()

# SVHN 中标签 10 表示数字 0
y_train[y_train == 10] = 0
y_test[y_test == 10] = 0

# 图像维度从 (32, 32, 3, N) 转成 (N, 32, 32, 3)
X_train = np.transpose(X_train_raw, (3, 0, 1, 2))
X_test = np.transpose(X_test_raw, (3, 0, 1, 2))

print("处理后 X_train shape:", X_train.shape)
print("处理后 y_train shape:", y_train.shape)
print("处理后 X_test shape:", X_test.shape)
print("处理后 y_test shape:", y_test.shape)

print("训练集类别分布：", np.bincount(y_train))
print("测试集类别分布：", np.bincount(y_test))

In [ ]:
# =========================
# 3. 可视化部分训练样本
# =========================
plt.figure(figsize=(10, 6))
for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(X_train[i])
    plt.title(f"Label: {y_train[i]}")
    plt.axis("off")
plt.suptitle("SVHN 训练集样本示例")
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# 4. 构建 Dataset
# =========================
class SVHNDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels.astype(np.int64)
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]

        if self.transform is not None:
            image = self.transform(image)

        return image, label

In [ ]:
# =========================
# 5. 数据增强 / 标准化
# =========================
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomCrop(32, padding=2),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5])
])

test_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5])
])

train_dataset = SVHNDataset(X_train, y_train, transform=train_transform)
test_dataset = SVHNDataset(X_test, y_test, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("训练批次数:", len(train_loader))
print("测试批次数:", len(test_loader))

In [ ]:
# =========================
# 6. 构建 CNN 模型
# =========================
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = SimpleCNN(num_classes=NUM_CLASSES).to(device)
print(model)

In [ ]:
# =========================
# 7. 损失函数与优化器
# =========================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [ ]:
# =========================
# 8. 训练 / 测试函数
# =========================
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

In [ ]:
# =========================
# 9. 模型训练
# =========================
train_losses = []
test_losses = []
train_accuracies = []
test_accuracies = []

best_test_acc = 0.0

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    test_losses.append(test_loss)
    train_accuracies.append(train_acc)
    test_accuracies.append(test_acc)

    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Test  Loss: {test_loss:.4f} | Test  Acc: {test_acc:.4f}")
    print("-" * 50)

    if test_acc > best_test_acc:
        best_test_acc = test_acc
        torch.save(model.state_dict(), MODEL_SAVE_PATH)

print("训练完成。")
print(f"最佳测试准确率: {best_test_acc:.4f}")
print(f"模型已保存到: {MODEL_SAVE_PATH}")

In [ ]:
# =========================
# 10. 绘制训练/测试准确率曲线
# =========================
epochs = range(1, NUM_EPOCHS + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs, train_accuracies, marker='o', label='Train Accuracy')
plt.plot(epochs, test_accuracies, marker='s', label='Test Accuracy')
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("训练准确率与测试准确率曲线")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# 11. 绘制训练/测试损失曲线
# =========================
plt.figure(figsize=(8, 5))
plt.plot(epochs, train_losses, marker='o', label='Train Loss')
plt.plot(epochs, test_losses, marker='s', label='Test Loss')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("训练损失与测试损失曲线")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# 12. 加载最佳模型并测试
# =========================
best_model = SimpleCNN(num_classes=NUM_CLASSES).to(device)
best_model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))

final_test_loss, final_test_acc = evaluate(best_model, test_loader, criterion, device)

print("=" * 60)
print("最终测试结果")
print("=" * 60)
print(f"Final Test Loss     : {final_test_loss:.4f}")
print(f"Final Test Accuracy : {final_test_acc:.4f}")

In [ ]:
# =========================
# 13. 展示测试集部分预测结果
# =========================
best_model.eval()

images_batch, labels_batch = next(iter(test_loader))
images_batch = images_batch.to(device)
labels_batch = labels_batch.to(device)

with torch.no_grad():
    outputs = best_model(images_batch)
    _, preds = torch.max(outputs, 1)

images_np = images_batch.cpu().numpy()
labels_np = labels_batch.cpu().numpy()
preds_np = preds.cpu().numpy()

images_np = np.transpose(images_np, (0, 2, 3, 1))
images_np = (images_np * 0.5) + 0.5
images_np = np.clip(images_np, 0, 1)

plt.figure(figsize=(12, 8))
for i in range(16):
    plt.subplot(4, 4, i + 1)
    plt.imshow(images_np[i])
    plt.title(f"T:{labels_np[i]} / P:{preds_np[i]}")
    plt.axis("off")
plt.suptitle("测试集部分预测结果")
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# 14. 自动输出简要结论
# =========================
print("=" * 60)
print("实验总结")
print("=" * 60)
print("1. 已完成 SVHN Format 2 数据集的读取与预处理；")
print("2. 已使用 CNN 模型完成 0~9 共 10 类数字分类任务；")
print("3. 已输出测试集准确率（Test Accuracy）；")
print("4. 已绘制训练/测试准确率曲线；")
print("5. 已绘制训练/测试损失曲线；")
print("6. 已保存最佳模型权重文件。")
print(f"\n当前最佳 Test Accuracy 为：{best_test_acc:.4f}")